# Notebook 3: Data Cleaning and Transformation

This notebook focuses on cleaning and transforming the Developer Survey dataset to prepare it for analysis and visualization in PowerBI. Building on the exploratory data analysis (EDA) from Notebook 2, we apply necessary and sensible transformations to a subset of relevant columns, addressing issues such as null values, inconsistent formats, and duplicates. The cleaned dataset will be loaded into a new PostgreSQL table called `clean_survey`, optimized for downstream use.

### Objectives
- Clean and standardize data in the specified relevant columns.
- Handle missing values and duplicates appropriately.
- Ensure data types and formats are suitable for PowerBI.
- Create and populate the `clean_survey` table in PostgreSQL.

## Setup

We import the required libraries for data manipulation and database interaction, then establish a connection to the PostgreSQL database using environment variables for security.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, Float, Boolean
from sqlalchemy_utils import database_exists, create_database
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'dev_survey_insights')
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Create the database engine
engine = create_engine(DATABASE_URL)
print('Database connection established successfully.')

Database connection established successfully.


## Data Loading

We load the dataset from the `raw_survey` table into a Pandas DataFrame for processing.

In [2]:
# Load data from the raw_survey table
query = "SELECT * FROM raw_survey"
df = pd.read_sql(query, engine)
print(f"Data loaded successfully. Rows: {len(df)}, Columns: {len(df.columns)}")

Data loaded successfully. Rows: 98855, Columns: 129


## Column Selection

We filter the dataset to include only the relevant columns specified for analysis and visualization in PowerBI.

In [3]:
# Define relevant columns
relevant_columns = [
    # Demographic
    'Country', 'Gender', 'Age', 'FormalEducation', 'RaceEthnicity',
    # Professional Experience
    'DevType', 'CompanySize', 'Employment', 'YearsCoding', 'YearsCodingProf',
    # Technologies and Tools
    'LanguageWorkedWith', 'LanguageDesireNextYear', 'DatabaseWorkedWith',
    'DatabaseDesireNextYear', 'PlatformWorkedWith', 'PlatformDesireNextYear',
    'FrameworkWorkedWith', 'FrameworkDesireNextYear', 'IDE', 'OperatingSystem',
    # Salary and Compensation
    'Salary', 'ConvertedSalary', 'SalaryType', 'CurrencySymbol',
    # Job Satisfaction
    'JobSatisfaction', 'CareerSatisfaction', 'HopeFiveYears',
    # Miscellaneous
    'OpenSource', 'StackOverflowVisit', 'StackOverflowHasAccount',
    'StackOverflowParticipate', 'AIDangerous', 'AIInteresting',
    'AIResponsible', 'AIFuture'
]

# Create a new DataFrame with only the relevant columns
df_clean = df[relevant_columns].copy()
print(f"Filtered to relevant columns. New shape: {df_clean.shape}")

Filtered to relevant columns. New shape: (98855, 35)


## Data Transformations

We apply transformations to each category of columns to ensure consistency and usability. Each transformation is documented with its purpose.

### Demographic Columns

- **Country**: Standardize text and handle nulls.
- **Gender**: Simplify to main categories and handle nulls.
- **Age**: Convert ranges to numeric midpoints.
- **FormalEducation**: Standardize categories.
- **RaceEthnicity**: Simplify and standardize.

In [4]:
# Country: Convert to title case and replace nulls with 'Unknown'
df_clean['Country'] = df_clean['Country'].str.title().fillna('Unknown')

# Gender: Simplify to Male, Female, Other, or Prefer not to say
def simplify_gender(gender):
    if pd.isna(gender):
        return 'Prefer not to say'
    gender = gender.lower()
    if 'male' in gender:
        return 'Male'
    elif 'female' in gender:
        return 'Female'
    else:
        return 'Other'
df_clean['Gender'] = df_clean['Gender'].apply(simplify_gender)

# Age: Convert ranges to midpoints and handle nulls
age_mapping = {
    'Under 18 years old': 16,
    '18 - 24 years old': 21,
    '25 - 34 years old': 29.5,
    '35 - 44 years old': 39.5,
    '45 - 54 years old': 49.5,
    '55 - 64 years old': 59.5,
    '65 years or older': 70
}
# Map to numeric values first, leaving unmapped as NaN
df_clean['Age'] = df_clean['Age'].map(age_mapping)
# Calculate the median of the numeric values after mapping
age_median = df_clean['Age'].median()
# Fill NaN with the median
df_clean['Age'] = df_clean['Age'].fillna(age_median)

df_clean['Age'] = df_clean['Age'].map(age_mapping).fillna(df_clean['Age'].median())

# FormalEducation: Standardize and handle nulls
df_clean['FormalEducation'] = df_clean['FormalEducation'].str.strip().fillna('Unknown')

# RaceEthnicity: Simplify to main categories and handle nulls
def simplify_race(race):
    if pd.isna(race):
        return 'Prefer not to say'
    return race.split(';')[0] if ';' in race else race
df_clean['RaceEthnicity'] = df_clean['RaceEthnicity'].apply(simplify_race)

print('Demographic columns transformed successfully.')

Demographic columns transformed successfully.


### Professional Experience Columns

- **DevType**: Keep as semicolon-separated text for flexibility.
- **CompanySize**: Convert to categorical ranges.
- **Employment**: Standardize categories.
- **YearsCoding** and **YearsCodingProf**: Convert ranges to midpoints.

In [5]:
# Professional Experience Columns

# DevType: Keep as is, replace nulls with 'Unknown'
df_clean['DevType'] = df_clean['DevType'].fillna('Unknown')

# CompanySize: Standardize ranges
company_size_mapping = {
    'Fewer than 10 employees': '1-9',
    '10 to 19 employees': '10-19',
    '20 to 99 employees': '20-99',
    '100 to 499 employees': '100-499',
    '500 to 999 employees': '500-999',
    '1,000 to 4,999 employees': '1000-4999',
    '5,000 to 9,999 employees': '5000-9999',
    '10,000 or more employees': '10000+'
}
df_clean['CompanySize'] = df_clean['CompanySize'].map(company_size_mapping).fillna('Unknown')

# Employment: Standardize and handle nulls
df_clean['Employment'] = df_clean['Employment'].str.strip().fillna('Unknown')

# YearsCoding and YearsCodingProf: Convert to midpoints
years_mapping = {
    '0-2 years': 1,
    '3-5 years': 4,
    '6-8 years': 7,
    '9-11 years': 10,
    '12-14 years': 13,
    '15-17 years': 16,
    '18-20 years': 19,
    '21-23 years': 22,
    '24-26 years': 25,
    '27-29 years': 28,
    '30 or more years': 35
}
# Map YearsCoding to numeric values and handle nulls with median
df_clean['YearsCoding'] = df_clean['YearsCoding'].map(years_mapping)
years_coding_median = df_clean['YearsCoding'].median()
df_clean['YearsCoding'] = df_clean['YearsCoding'].fillna(years_coding_median)

# Map YearsCodingProf to numeric values, default to 0 for nulls (non-professionals)
df_clean['YearsCodingProf'] = df_clean['YearsCodingProf'].map(years_mapping).fillna(0)

print('Professional experience columns transformed successfully.')

Professional experience columns transformed successfully.


### Technologies and Tools Columns

- **LanguageWorkedWith**, **LanguageDesireNextYear**, etc.: Retain semicolon-separated format, handle nulls.
- **OperatingSystem**: Standardize categories.

In [6]:
# Technologies: Replace nulls with 'None'
tech_columns = ['LanguageWorkedWith', 'LanguageDesireNextYear', 'DatabaseWorkedWith',
                'DatabaseDesireNextYear', 'PlatformWorkedWith', 'PlatformDesireNextYear',
                'FrameworkWorkedWith', 'FrameworkDesireNextYear', 'IDE']
for col in tech_columns:
    df_clean[col] = df_clean[col].fillna('None')

# OperatingSystem: Standardize and handle nulls
df_clean['OperatingSystem'] = df_clean['OperatingSystem'].str.strip().fillna('Unknown')

print('Technologies and tools columns transformed successfully.')

Technologies and tools columns transformed successfully.


### Salary and Compensation Columns

- **Salary**: Clean non-numeric values, keep as string.
- **ConvertedSalary**: Cap outliers and handle nulls.
- **SalaryType** and **CurrencySymbol**: Standardize and handle nulls.

In [7]:
# Salary: Convert to string, handle nulls
df_clean['Salary'] = df_clean['Salary'].astype(str).replace('nan', 'Unknown')

# ConvertedSalary: Cap at 99th percentile and impute nulls with median
salary_99th = df_clean['ConvertedSalary'].quantile(0.99)
df_clean['ConvertedSalary'] = df_clean['ConvertedSalary'].clip(upper=salary_99th)
df_clean['ConvertedSalary'] = df_clean['ConvertedSalary'].fillna(df_clean['ConvertedSalary'].median())

# SalaryType and CurrencySymbol: Handle nulls
df_clean['SalaryType'] = df_clean['SalaryType'].fillna('Unknown')
df_clean['CurrencySymbol'] = df_clean['CurrencySymbol'].fillna('Unknown')

print('Salary and compensation columns transformed successfully.')

Salary and compensation columns transformed successfully.


### Job Satisfaction Columns

- **JobSatisfaction**, **CareerSatisfaction**, **HopeFiveYears**: Standardize and handle nulls.

In [8]:
# Job Satisfaction: Standardize and handle nulls
satisfaction_columns = ['JobSatisfaction', 'CareerSatisfaction', 'HopeFiveYears']
for col in satisfaction_columns:
    df_clean[col] = df_clean[col].str.strip().fillna('Unknown')

print('Job satisfaction columns transformed successfully.')

Job satisfaction columns transformed successfully.


### Miscellaneous Columns

- **OpenSource**: Convert to boolean where possible.
- **StackOverflowVisit**, etc.: Standardize and handle nulls.
- **AI-related columns**: Standardize and handle nulls.

In [9]:
# OpenSource: Convert to boolean
df_clean['OpenSource'] = df_clean['OpenSource'].map({'Yes': True, 'No': False}).fillna(False)

# StackOverflow columns: Standardize and handle nulls
so_columns = ['StackOverflowVisit', 'StackOverflowHasAccount', 'StackOverflowParticipate']
for col in so_columns:
    df_clean[col] = df_clean[col].str.strip().fillna('Unknown')

# AI columns: Standardize and handle nulls
ai_columns = ['AIDangerous', 'AIInteresting', 'AIResponsible', 'AIFuture']
for col in ai_columns:
    df_clean[col] = df_clean[col].str.strip().fillna('Unknown')

print('Miscellaneous columns transformed successfully.')

Miscellaneous columns transformed successfully.


## Null Handling

Most nulls have been addressed in the transformations above. We verify the remaining null counts to ensure completeness.

In [10]:
# Check for remaining nulls
null_counts = df_clean.isnull().sum()
print('Remaining null counts:\n', null_counts[null_counts > 0])

Remaining null counts:
 Series([], dtype: int64)


## Duplicate Check  

We could check for and remove duplicate rows. However, since this is a sample with a specifically selected set of columns, there isn't much criteria to accurately determine whether a row is truly duplicated data or if different individuals coincidentally provided the same answers. Because of that, we are not removing duplicate rows.

In [11]:
'''
# Check for duplicates
duplicates = df_clean.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')

# Show a sample of duplicate rows to inspect
if duplicates > 0:
    df_duplicates = df_clean[df_clean.duplicated(keep=False)]
    print('Sample of duplicate rows (first 5):')
    print(df_duplicates.head())
    
    # Keep the first instance of each duplicate and remove the rest
    df_clean = df_clean.drop_duplicates(keep='first')
    print(f'Duplicates removed. New shape: {df_clean.shape}')
else:
    print('No duplicates found.')
'''

"\n# Check for duplicates\nduplicates = df_clean.duplicated().sum()\nprint(f'Number of duplicate rows: {duplicates}')\n\n# Show a sample of duplicate rows to inspect\nif duplicates > 0:\n    df_duplicates = df_clean[df_clean.duplicated(keep=False)]\n    print('Sample of duplicate rows (first 5):')\n    print(df_duplicates.head())\n    \n    # Keep the first instance of each duplicate and remove the rest\n    df_clean = df_clean.drop_duplicates(keep='first')\n    print(f'Duplicates removed. New shape: {df_clean.shape}')\nelse:\n    print('No duplicates found.')\n"

## Table Creation

We define and create the `clean_survey` table in PostgreSQL with appropriate data types for each column.

In [12]:
# Define metadata
metadata = MetaData()

# Define clean_survey table
clean_survey = Table(
    'clean_survey', metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    # Demographic
    Column('Country', String),
    Column('Gender', String),
    Column('Age', Float),
    Column('FormalEducation', String),
    Column('RaceEthnicity', String),
    # Professional Experience
    Column('DevType', String),
    Column('CompanySize', String),
    Column('Employment', String),
    Column('YearsCoding', Float),
    Column('YearsCodingProf', Float),
    # Technologies and Tools
    Column('LanguageWorkedWith', String),
    Column('LanguageDesireNextYear', String),
    Column('DatabaseWorkedWith', String),
    Column('DatabaseDesireNextYear', String),
    Column('PlatformWorkedWith', String),
    Column('PlatformDesireNextYear', String),
    Column('FrameworkWorkedWith', String),
    Column('FrameworkDesireNextYear', String),
    Column('IDE', String),
    Column('OperatingSystem', String),
    # Salary and Compensation
    Column('Salary', String),
    Column('ConvertedSalary', Float),
    Column('SalaryType', String),
    Column('CurrencySymbol', String),
    # Job Satisfaction
    Column('JobSatisfaction', String),
    Column('CareerSatisfaction', String),
    Column('HopeFiveYears', String),
    # Miscellaneous
    Column('OpenSource', Boolean),
    Column('StackOverflowVisit', String),
    Column('StackOverflowHasAccount', String),
    Column('StackOverflowParticipate', String),
    Column('AIDangerous', String),
    Column('AIInteresting', String),
    Column('AIResponsible', String),
    Column('AIFuture', String)
)

# Create the table
metadata.create_all(engine)
print('Table clean_survey created successfully.')

Table clean_survey created successfully.


## Data Loading

We load the cleaned DataFrame into the `clean_survey` table, replacing any existing data.

In [ ]:
# Load the cleaned data into clean_survey
df_clean.to_sql('clean_survey', engine, if_exists='replace', index=False, method='multi')
print('Cleaned data loaded into clean_survey table successfully.')

## Verification

We query the first 5 rows of the `clean_survey` table to confirm the data was loaded correctly.

In [1]:
# Verify the data
query = "SELECT * FROM clean_survey LIMIT 5;"
sample = pd.read_sql(query, engine)
print('Sample of loaded data:\n', sample)

NameError: name 'pd' is not defined